# Manuscript analysis from published aggregate results

**Beyond Rigid Guidelines: Learning the Nuance of Real-World Breast Cancer Imaging Surveillance**

Run all cells after installing `requirements-analysis.txt`. No ConcertAI license, patient records, model checkpoints, GPU, or training run is needed for this notebook. It reads only the six included aggregate CSV files and reproduces the published metric tables and Figures 2 and 3.

These are saved study results: displaying confidence intervals does **not** rerun the patient bootstrap, and this notebook does **not** independently recompute metrics from predictions. Training and patient-level evaluation require your own authorized data. See `DATA_AVAILABILITY.md`.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / 'aggregate_results' / 'table_ii_metrics.csv').is_file()
             and (p / 'src' / 'analysis' / 'plot_baseline_comparisons.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook from the release repository or its notebooks directory.')
AGGREGATES = ROOT / 'aggregate_results'
OUTPUT = ROOT / 'generated_figures'
OUTPUT.mkdir(exist_ok=True)
sys.path.insert(0, str(ROOT / 'src' / 'analysis'))
from plot_baseline_comparisons import load_guideline_rows, load_average_rows, make_vertical_figure
from plot_feature_contributions import plot_feature_group_contribution

names = ['table_ii_metrics', 'table_iii_subgroups', 'xgboost_dsi_per_scan_type_f1',
         'guideline_comparison', 'average_comparison', 'feature_group_contributions']
tables = {name: pd.read_csv(AGGREGATES / f'{name}.csv') for name in names}
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_rows', 60)


## Table II: model and feature-set comparisons

D = demographics, S = staging, I = imaging history, M = mastectomy history, B = biomarker history. AUC and F1 are higher-is-better; calibration error (ECE), MAE, and RMSE are lower-is-better. Interval errors are in months. The saved lower/upper CI columns are patient-bootstrap 95% confidence intervals.


In [ ]:
table_ii = tables['table_ii_metrics']
assert len(table_ii) == 48, 'Expected six models and eight feature sets.'
assert not table_ii.duplicated(['model', 'feature_set']).any()
display(table_ii)


## Table III: scan-type and stage-trajectory subgroups

Subgroup counts and percentages have the denominators used in the manuscript; scan-type groups can overlap because the task is multilabel.

In [ ]:
for model, rows in tables['table_iii_subgroups'].groupby('model', sort=False):
    display(Markdown(f'### {model}'))
    display(rows.reset_index(drop=True))


## XGBoost-DSI: per-scan-type precision, recall, and F1

In [ ]:
display(tables['xgboost_dsi_per_scan_type_f1'])


## Figure 2: interval baseline comparisons

The **Nearest guideline** ultrasound comparator chooses the closest interval using the observed target. It is an oracle comparator, not a prospective policy; fixed 6-, 12-, and 24-month comparators are shown separately. This notebook preserves the manuscript comparison rather than treating the oracle result as leakage-free.


In [ ]:
display(tables['guideline_comparison'])
display(tables['average_comparison'])
figure = make_vertical_figure(
    load_guideline_rows(AGGREGATES / 'guideline_comparison.csv'),
    load_average_rows(AGGREGATES / 'average_comparison.csv'),
)
figure.savefig(OUTPUT / 'figure_2_baseline_comparisons.pdf', bbox_inches='tight')
display(figure)
plt.close(figure)


## Figure 3: feature-group contributions

Contributions summarize the saved model comparisons; they are not causal effects.

In [ ]:
display(tables['feature_group_contributions'])
figure = plot_feature_group_contribution(tables['feature_group_contributions'])
figure.savefig(OUTPUT / 'figure_3_feature_group_contributions.pdf', bbox_inches='tight')
display(figure)
plt.close(figure)
print('Saved Figure 2 and Figure 3 to generated_figures/.')
